# SAAM Project - Pacific Region

Sustainable & Active Asset Management - portfolio construction with carbon constraints.

This notebook is fully self-contained: it reads the project input files directly from `data/` and produces every table in `outputs/tables/` and every figure in `outputs/figures/`. It does **not** import anything from `src/`. The methodology follows the project PDF and the existing src reference code (MVP-construction, 05_part3.py, 06_part4.py).

Structure:
1. Setup
2. Short introduction
3. Data loading and cleaning
4. Part 1 - Minimum-Variance Portfolio (P_mv)
5. Part 2 - Value-Weighted benchmark (P_vw)
6. Part 3 - 50% carbon-footprint reduction (P_mv_50, P_vw_50)
7. Part 4 - Net-Zero trajectory (P_vw_NZ)
8. Final performance summary
9. Short conclusion

## 1. Setup

Imports, robust pathlib paths, methodology constants, and output folders.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# project paths
ROOT = Path.cwd()
if ROOT.name == "src":
    ROOT = ROOT.parent
DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
OUTPUTS = ROOT / "outputs"
TABLES = OUTPUTS / "tables"
FIGURES = OUTPUTS / "figures"
for d in (OUTPUTS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# methodology constants (identical to the src reference)
YEAR_FIRST = 2013      # first allocation year (decision at end of Y)
YEAR_LAST = 2024       # last allocation year -> implementation Y+1 = 2025
PERF_START = 2014      # first implementation year (inclusive)
PERF_END = 2025        # last implementation year (inclusive)
ESTIM_MONTHS = 120     # 10-year rolling estimation window
MIN_OBS = 36           # >= 3 years of returns in the window
STALE_THRESHOLD = 0.50 # <= 50% zero returns in the window
RIDGE_EPS = 1e-6       # ridge added to Sigma
THETA_NZ = 0.10        # 10% per year carbon decay (Part 4)
CARBON_SCOPE = "scope1" # assigned climate strategy for this group

pd.options.display.float_format = "{:,.4f}".format
print("Setup OK. ROOT =", ROOT)

def file_inventory(folder):
    """Return a notebook-friendly inventory of files under a folder."""
    rows = []
    for path in sorted(folder.rglob("*")):
        if path.is_file():
            rows.append({
                "folder": str(path.parent.relative_to(ROOT)),
                "file": path.name,
                "relative_path": str(path.relative_to(ROOT)),
                "size_kb": path.stat().st_size / 1024.0,
            })
    return pd.DataFrame(rows)

input_files = pd.concat([file_inventory(RAW), file_inventory(PROCESSED)], ignore_index=True)
print("Input files available to this notebook:")
display(input_files)

## 2. Short introduction

We build five portfolios on the Pacific equity universe and compare their financial **and** carbon performance over 2014-2025:

- **P_vw**: market-cap weighted benchmark, monthly rebalanced (Section 2.3).
- **P_mv**: long-only minimum-variance portfolio, annually rebalanced, intra-year drift (Section 2.2).
- **P_mv_50**: minimum variance with carbon footprint capped at 50% of P_mv (Section 3.2).
- **P_vw_50**: tracking-error minimisation vs P_vw with the same 50% CF cap (Section 3.3).
- **P_vw_NZ**: tracking-error minimisation vs P_vw with a declining Net-Zero cap, -10% per year from the 2013 anchor (Section 4).

Carbon metrics:
- WACI = $\sum_i w_i\, \text{CI}_i$ with $\text{CI}_i = E_i / (\text{Rev}_i / 1000)$ (tCO2e per M$ revenue).
- Carbon Footprint CF = $\sum_i w_i \, (E_i / \text{Cap}_i)$ (tCO2e per M$ invested).

Why it matters: investors want low risk and exposure to the climate transition. Comparing constrained portfolios to unconstrained ones shows the financial *cost*, or *absence of cost*, of decarbonisation.

## 3. Data loading and cleaning

We read the Pacific input panels: prices, emissions (Scope 1), revenues, market cap (annual + monthly), risk-free rate. PDF Section 2.1 cleaning rules are applied to prices:
- price < 0.5 -> NaN,
- internal gaps -> forward fill,
- trailing NaNs (delisting) -> 0 so the delisting-month return = -100%.

Annual panels (CO2, revenues, cap_y) are forward-filled to plug rare missing entries.

In [ ]:
# generic loader for Datastream-style wide CSVs (ISIN x periods)
def read_wide(path, date_columns):
    df = pd.read_csv(path)
    df = df[df["ISIN"].notna()].copy()
    df["ISIN"] = df["ISIN"].astype(str)
    df = df.set_index("ISIN").drop(columns=["NAME"], errors="ignore")
    df = df.apply(pd.to_numeric, errors="coerce")
    if date_columns:
        df.columns = pd.to_datetime(df.columns).normalize()
    else:
        df.columns = df.columns.astype(int)
    return df

# read panels (already cleaned at ingest stage)
prices_wide = read_wide(PROCESSED / "Clean_Prices_Pacific.csv", date_columns=True)
emissions = read_wide(PROCESSED / "Clean_CO2_Scope1_Pacific.csv", date_columns=False).sort_index(axis=1).ffill(axis=1)
revenues = read_wide(PROCESSED / "Clean_Revenues_Pacific.csv", date_columns=False).sort_index(axis=1).ffill(axis=1)

static = pd.read_csv(PROCESSED / "Pacific_Universe.csv")
name_col = "NAME" if "NAME" in static.columns else "Name"
names = static.set_index("ISIN")[name_col].astype(str)
countries = static.set_index("ISIN")["Country"].astype(str)
pacific = set(static["ISIN"].astype(str))

# market cap (Y for eligibility/CF, M for the VW benchmark)
cap_y = read_wide(RAW / "DS_MV_T_USD_Y.csv", date_columns=False).sort_index(axis=1).ffill(axis=1)
cap_y = cap_y.loc[cap_y.index.intersection(pacific)]
cap_m = read_wide(RAW / "DS_MV_T_USD_M.csv", date_columns=True).T.sort_index()
cap_m = cap_m.loc[:, cap_m.columns.intersection(list(pacific))]

print("Loaded:", "prices", prices_wide.shape, "| emissions", emissions.shape,
      "| revenues", revenues.shape, "| cap_y", cap_y.shape, "| cap_m", cap_m.shape)

In [ ]:
# price cleaning (Section 2.1)
prices = prices_wide.T.sort_index()                # dates x ISIN
prices = prices.where(prices >= 0.5)               # <0.5 -> NaN
has_val = prices.notna().astype(np.int8)
ahead = has_val.iloc[::-1].cumsum(axis=0).iloc[::-1]
trailing = prices.isna() & (ahead == 0)            # delisting
prices = prices.ffill(axis=0)
prices[trailing] = 0.0

# monthly returns (delisting handled as -100%)
prev = prices.shift(1)
rets = prices / prev.replace(0.0, np.nan) - 1.0
rets[(prices == 0.0) & (prev > 0.0)] = -1.0
rets = rets.replace([np.inf, -np.inf], np.nan)

# align all panels on the common ISINs
common = (prices.columns
          .intersection(emissions.index)
          .intersection(revenues.index)
          .intersection(cap_y.index))
prices = prices[common]
rets = rets[common]
emissions = emissions.loc[common]
revenues = revenues.loc[common]
cap_y = cap_y.loc[common]
cap_m = cap_m.loc[:, cap_m.columns.intersection(common)]

# Pacific monthly RF (period column = YYYYMM int, RF in % per month)
rf_raw = pd.read_csv(PROCESSED / "rf_rate.csv")
rf_raw.columns = ["period", "rf"]
rf_raw = rf_raw.dropna()
rf_dates = pd.to_datetime(rf_raw["period"].astype(int).astype(str).str.zfill(6),
                          format="%Y%m") + pd.offsets.MonthEnd(0)
rf_monthly = pd.Series(rf_raw["rf"].to_numpy() / 100.0,
                       index=rf_dates, name="rf").sort_index()

rf_timeline = rf_monthly.loc["2013-12-31":"2024-12-31"].rename("monthly_rf_decimal").to_frame()
rf_timeline["monthly_rf_percent"] = rf_timeline["monthly_rf_decimal"] * 100.0
rf_timeline.index.name = "date"
rf_timeline.to_csv(TABLES / "risk_free_rate_dec2013_dec2024.csv")

print(f"Aligned universe: {len(common)} ISINs | rets {rets.shape} | rf_monthly {rf_monthly.shape}")
print(f"Risk-free timeline Dec 2013-Dec 2024: {len(rf_timeline)} months; saved to outputs/tables/risk_free_rate_dec2013_dec2024.csv")
display(rf_timeline)

### Helper functions

Three reusable helpers (faithful to the src code):
- `build_year_inputs(Y)`: eligible universe, complete-case covariance, value weights, carbon vectors at end-of-Y.
- `solve_qp(...)`: long-only QP (variance or tracking error) with an optional carbon-footprint cap, SLSQP with analytic gradients and the same warm-start trick used in src so the cap is always feasible.
- `simulate_year(weights, returns)`: monthly portfolio returns over the implementation year, weights drift (no intra-year rebalancing).

In [ ]:
def build_year_inputs(year):
    est_start = pd.Timestamp(year - 9, 1, 1)
    est_end = pd.Timestamp(year, 12, 31)
    oos_start = pd.Timestamp(year + 1, 1, 1)
    oos_end = pd.Timestamp(year + 1, 12, 31)

    rets_win = rets.loc[est_start:est_end]
    rets_oos = rets.loc[oos_start:oos_end]

    obs = rets_win.count()
    zero_ratio = (rets_win == 0.0).sum() / obs.replace(0, np.nan)
    last_px_ok = prices.loc[:est_end].iloc[-1].gt(0)

    # eligibility filters: carbon, revenue, cap > 0 at Y; price > 0; >=36 obs; <=50% zeros
    eligible = (
        emissions[year].notna()
        & revenues[year].gt(0)
        & cap_y[year].gt(0)
        & last_px_ok.reindex(emissions.index).fillna(False)
        & obs.reindex(emissions.index).ge(MIN_OBS).fillna(False)
        & zero_ratio.reindex(emissions.index).le(STALE_THRESHOLD).fillna(False)
    )
    isins = sorted(eligible[eligible].index.intersection(rets_oos.columns))
    if not isins:
        raise RuntimeError(f"No eligible firms for year {year}")

    rets_est = rets_win[isins].iloc[-ESTIM_MONTHS:]
    complete = rets_est.dropna(how="any")
    if len(complete) < MIN_OBS:
        filled = rets_est.apply(lambda s: s.fillna(s.mean())).fillna(0.0)
        cov = np.cov(filled.to_numpy(), rowvar=False, ddof=0)
    else:
        cov = np.cov(complete.to_numpy(), rowvar=False, ddof=0)
    cov = (cov + cov.T) / 2.0 + RIDGE_EPS * np.eye(len(isins))

    cap_yi = cap_y.loc[isins, year].astype(float)
    emi_yi = emissions.loc[isins, year].astype(float)
    rev_yi = revenues.loc[isins, year].astype(float)
    return {
        "year": year, "isins": isins,
        "rets_oos": rets_oos[isins],
        "cov": cov,
        "ci": emi_yi / (rev_yi / 1000.0),       # tCO2e per M$ revenue
        "cf_per_w": emi_yi / cap_yi,            # tCO2e per M$ invested (per unit weight)
        "w_vw": (cap_yi / cap_yi.sum()).to_numpy(),
    }

In [ ]:
def solve_qp(cov, objective="variance", benchmark=None, carbon=None, carbon_limit=None):
    """Long-only QP. SLSQP with analytic gradients.
    min w'Sigma w  (variance)  OR  min (w-b)'Sigma(w-b)  (tracking_error)
    s.t. sum(w)=1, 0<=w<=1, optional carbon.w <= carbon_limit."""
    n = cov.shape[0]
    bench = np.full(n, 1.0 / n) if benchmark is None else np.asarray(benchmark, float)
    x0 = bench.copy() if objective == "tracking_error" else np.full(n, 1.0 / n)

    # if warm-start violates the cap, pull it towards the lowest-carbon asset
    if carbon is not None and carbon_limit is not None and float(x0 @ carbon) > carbon_limit:
        unit = np.zeros(n); unit[int(np.argmin(carbon))] = 1.0
        base, low = float(x0 @ carbon), float(unit @ carbon)
        if low > carbon_limit:
            x0 = unit
        else:
            alpha = float(np.clip((base - carbon_limit) / max(base - low, 1e-12) + 1e-3, 0.0, 1.0))
            x0 = (1.0 - alpha) * x0 + alpha * unit

    if objective == "variance":
        fun = lambda w: float(w @ cov @ w)
        jac = lambda w: 2.0 * cov @ w
    else:
        b = bench
        fun = lambda w: float((w - b) @ cov @ (w - b))
        jac = lambda w: 2.0 * cov @ (w - b)

    cons = [{"type": "eq", "fun": lambda w: float(w.sum() - 1.0),
             "jac": lambda w: np.ones_like(w)}]
    if carbon is not None and carbon_limit is not None:
        c = np.asarray(carbon, float); L = float(carbon_limit)
        cons.append({"type": "ineq",
                     "fun": lambda w, c=c, L=L: float(L - w @ c),
                     "jac": lambda w, c=c: -c})

    # ftol=1e-8 / maxiter=200 is plenty for the SAAM QPs and ~3x faster than 1e-10
    res = minimize(fun, x0, jac=jac, method="SLSQP", bounds=[(0.0, 1.0)] * n,
                   constraints=cons,
                   options={"maxiter": 200, "ftol": 1e-8, "disp": False})
    w = np.clip(res.x, 0.0, None)
    if w.sum() > 0:
        w = w / w.sum()
    ok = bool(res.success)
    if carbon is not None and carbon_limit is not None:
        ok = ok and float(w @ np.asarray(carbon, float)) <= carbon_limit + 1e-6
    return w, ok, str(res.message)


def simulate_year(weights, rets_oos):
    """Monthly returns of a portfolio with drifting weights over the implementation year."""
    w = weights.copy()
    out = {}
    for date, row in rets_oos.fillna(0.0).iterrows():
        r = row.to_numpy(float)
        rp = float(w @ r)
        out[date] = rp
        if 1.0 + rp <= 0.0:
            break
        w = np.clip(w * (1.0 + r) / (1.0 + rp), 0.0, None)
        if w.sum() > 0:
            w = w / w.sum()
    return pd.Series(out, dtype=float)


def vw_monthly_rebalanced():
    """Strict Section 2.3 VW benchmark using previous-period weights."""
    cap = cap_m.reindex(index=rets.index, columns=rets.columns)
    w = cap.div(cap.sum(axis=1), axis=0).shift(1)
    return (w * rets).sum(axis=1, min_count=1)

### Annual loop: solve every portfolio in a single pass

We loop once over Y = 2013..2024 and solve all five portfolios on the *same* Sigma_Y. This avoids the duplicate work that the src code did (Parts 3 and 4 ran two separate annual passes). Inside the loop we collect:
- monthly returns (drift) for vw_drift / mv / mv_50 / vw_50 / vw_nz,
- weights at the start of each implementation year (positive weights only),
- annual carbon metrics (CF, WACI) and constraint slack,
- ex-ante / ex-post annualised tracking error vs the VW benchmark,
- top-10 WACI / CF contributors of the VW benchmark.

In [ ]:
# anchor footprint CF(P_vw) at Y0 = YEAR_FIRST (used for the NZ cap path)
yi_anchor = build_year_inputs(YEAR_FIRST)
cf_vw_anchor = float(yi_anchor["w_vw"] @ yi_anchor["cf_per_w"].to_numpy())
print(f"Anchor CF(P_vw)_{YEAR_FIRST} = {cf_vw_anchor:.4f} tCO2e/M$")

labels = ["vw_drift", "mv", "mv_50", "vw_50", "vw_nz"]
month_segs = {lab: [] for lab in labels}
annual_rows, weight_rows, te_rows, slack_rows = [], [], [], []
waci_top_rows, cf_top_rows, path_rows = [], [], []

for year in range(YEAR_FIRST, YEAR_LAST + 1):
    yi = build_year_inputs(year)
    carbon = yi["cf_per_w"].to_numpy()
    ci_vec = yi["ci"].to_numpy()
    w_vw = yi["w_vw"]
    cf_vw = float(w_vw @ carbon)

    # solve four QPs on the same Sigma_Y
    w_mv,   ok_mv,   msg_mv   = solve_qp(yi["cov"], "variance")
    cf_mv = float(w_mv @ carbon)
    w_mv50, ok_mv50, msg_mv50 = solve_qp(yi["cov"], "variance",
                                          carbon=carbon, carbon_limit=0.5 * cf_mv)
    w_vw50, ok_vw50, msg_vw50 = solve_qp(yi["cov"], "tracking_error", benchmark=w_vw,
                                          carbon=carbon, carbon_limit=0.5 * cf_vw)
    cap_nz = ((1.0 - THETA_NZ) ** (year - YEAR_FIRST + 1)) * cf_vw_anchor
    w_nz,   ok_nz,   msg_nz   = solve_qp(yi["cov"], "tracking_error", benchmark=w_vw,
                                          carbon=carbon, carbon_limit=cap_nz)

    portfolios = {
        "vw_drift": (w_vw,   True,    "benchmark (drift)", cf_vw),
        "mv":       (w_mv,   ok_mv,   msg_mv,              np.nan),
        "mv_50":    (w_mv50, ok_mv50, msg_mv50,            0.5 * cf_mv),
        "vw_50":    (w_vw50, ok_vw50, msg_vw50,            0.5 * cf_vw),
        "vw_nz":    (w_nz,   ok_nz,   msg_nz,              cap_nz),
    }

    # simulate implementation year for each portfolio
    bench_series = None
    for lab, (w, ok, msg, cap_lim) in portfolios.items():
        s = simulate_year(w, yi["rets_oos"])
        month_segs[lab].append(s)
        if lab == "vw_drift":
            bench_series = s

        cf = float(w @ carbon)
        wc = float(w @ ci_vec)
        cap_val = None if (isinstance(cap_lim, float) and np.isnan(cap_lim)) else cap_lim
        annual_rows.append({
            "year": year, "implementation_year": year + 1, "portfolio": lab,
            "n_assets": len(yi["isins"]),
            "waci_tco2e_per_musd_revenue": wc,
            "carbon_footprint_tco2e_per_musd_invested": cf,
            "carbon_limit": cap_lim,
            "optimization_success": bool(ok), "optimization_message": msg,
        })
        slack_rows.append({
            "year": year, "implementation_year": year + 1, "portfolio": lab,
            "carbon_footprint": cf,
            "carbon_limit": cap_val if cap_val is not None else np.nan,
            "slack": (cap_val - cf) if cap_val is not None else np.nan,
            "satisfied_within_1e-6": bool(cap_val is None or (cap_val - cf) >= -1e-6),
            "optimization_success": bool(ok), "optimization_message": msg,
        })

        active = pd.Series(w, index=yi["isins"])
        for isin, weight in active[active > 1e-6].sort_values(ascending=False).items():
            weight_rows.append({
                "year": year, "portfolio": lab, "ISIN": isin,
                "name": names.get(isin, ""), "country": countries.get(isin, ""),
                "weight": float(weight),
                "ci": float(yi["ci"].get(isin, np.nan)),
                "cf_per_weight": float(yi["cf_per_w"].get(isin, np.nan)),
            })

        diff_w = w - w_vw
        ex_ante = float(np.sqrt(max(diff_w @ yi["cov"] @ diff_w, 0.0) * 12.0))
        ex_post = np.nan
        if lab != "vw_drift" and bench_series is not None:
            rel = (s.reindex(bench_series.index) - bench_series).dropna()
            if len(rel) > 1:
                ex_post = float(rel.std(ddof=0) * np.sqrt(12.0))
        te_rows.append({
            "year": year, "implementation_year": year + 1, "portfolio": lab,
            "ex_ante_tracking_error_annual": ex_ante,
            "ex_post_tracking_error_annual": ex_post,
        })

    # top-10 WACI / CF contributors using VW weights
    contrib = pd.DataFrame({
        "ISIN": yi["isins"],
        "name": names.reindex(yi["isins"]).to_numpy(),
        "country": countries.reindex(yi["isins"]).to_numpy(),
        "ci_tco2e_per_musd_revenue": ci_vec,
        "cf_per_weight_tco2e_per_musd_invested": carbon,
        "vw_weight": w_vw,
        "vw_waci_contribution": w_vw * ci_vec,
        "vw_cf_contribution": w_vw * carbon,
    })
    top_waci = contrib.sort_values("vw_waci_contribution", ascending=False).head(10)
    for rank, row in enumerate(top_waci.itertuples(index=False), start=1):
        waci_top_rows.append({
            "year": year, "rank": rank, "ISIN": row.ISIN, "name": row.name,
            "country": row.country,
            "ci_tco2e_per_musd_revenue": row.ci_tco2e_per_musd_revenue,
            "vw_weight": row.vw_weight,
            "vw_waci_contribution": row.vw_waci_contribution,
        })
    top_cf = contrib.sort_values("vw_cf_contribution", ascending=False).head(10)
    for rank, row in enumerate(top_cf.itertuples(index=False), start=1):
        cf_top_rows.append({
            "year": year, "rank": rank, "ISIN": row.ISIN, "name": row.name,
            "country": row.country,
            "cf_per_weight_tco2e_per_musd_invested": row.cf_per_weight_tco2e_per_musd_invested,
            "vw_weight": row.vw_weight,
            "vw_cf_contribution": row.vw_cf_contribution,
        })

    # NZ cap-path row
    cf_vw_nz = float(w_nz @ carbon)
    path_rows.append({
        "year": year, "implementation_year": year + 1,
        "anchor_cf_vw_2013": cf_vw_anchor, "theta": THETA_NZ,
        "carbon_limit_nz": cap_nz,
        "carbon_footprint_vw": cf_vw,
        "carbon_footprint_vw_nz": cf_vw_nz,
        "slack": cap_nz - cf_vw_nz,
        "feasible_within_1e-6": bool((cap_nz - cf_vw_nz) >= -1e-6),
        "optimizer_success": bool(ok_nz),
    })

    print(f"{year}: n={len(yi['isins']):>3} | CF vw={cf_vw:7.2f}  mv={cf_mv:7.2f}  "
          f"mv50={float(w_mv50@carbon):7.2f}  vw50={float(w_vw50@carbon):7.2f}  "
          f"vw_nz={cf_vw_nz:7.2f} (cap={cap_nz:6.2f})")

print("\nAnnual loop done.")

In [ ]:
# assemble the joint monthly panel and restrict to performance window
monthly = pd.DataFrame({lab: pd.concat(month_segs[lab]).sort_index() for lab in labels})
monthly = monthly.loc[f"{PERF_START}-01-01":f"{PERF_END}-12-31"]

# strict monthly-rebalanced VW benchmark (Section 2.3)
vw_monthly = vw_monthly_rebalanced().loc[f"{PERF_START}-01-01":f"{PERF_END}-12-31"]
monthly["vw"] = vw_monthly.reindex(monthly.index)
monthly = monthly[["vw", "vw_drift", "mv", "mv_50", "vw_50", "vw_nz"]]

annual = pd.DataFrame(annual_rows)
weights = pd.DataFrame(weight_rows)
te_table = pd.DataFrame(te_rows)
slack_table = pd.DataFrame(slack_rows)
waci_top = pd.DataFrame(waci_top_rows)
cf_top = pd.DataFrame(cf_top_rows)
path_table = pd.DataFrame(path_rows)

# The annual loop used the annual VW allocation as the optimization benchmark.
# For reported ex-post tracking error, compare realised returns with the PDF's monthly-rebalanced VW benchmark.
for idx, row in te_table.iterrows():
    lab = row["portfolio"]
    if lab == "vw_drift" or lab not in monthly.columns:
        continue
    year = int(row["implementation_year"])
    rel = (monthly.loc[f"{year}-01-01":f"{year}-12-31", lab]
           - monthly.loc[f"{year}-01-01":f"{year}-12-31", "vw"]).dropna()
    te_table.at[idx, "ex_post_tracking_error_annual"] = float(rel.std(ddof=0) * np.sqrt(12.0)) if len(rel) > 1 else np.nan

# combined files with every constructed portfolio in one place
monthly.to_csv(TABLES / "all_portfolio_monthly_returns.csv", index_label="date")
annual.to_csv(TABLES / "all_portfolio_annual_carbon_metrics.csv", index=False)
weights.to_csv(TABLES / "all_portfolio_weights.csv", index=False)
te_table.to_csv(TABLES / "all_portfolio_tracking_errors.csv", index=False)
slack_table.to_csv(TABLES / "all_portfolio_constraint_slack.csv", index=False)

# sanity check: no carbon-constraint violations
constrained = slack_table[slack_table["carbon_limit"].notna() & (slack_table["portfolio"] != "vw_drift")]
worst_slack = constrained["slack"].min()
assert worst_slack >= -1e-6, f"Carbon constraint violated: min slack = {worst_slack}"
print(f"Monthly panel: {monthly.shape} | min carbon slack = {worst_slack:.3e}")
print("Constructed portfolios:", ", ".join(monthly.columns))
print("Combined portfolio files saved: all_portfolio_monthly_returns.csv, all_portfolio_weights.csv, all_portfolio_annual_carbon_metrics.csv")
monthly.head()

In [ ]:
# summary statistics with annualized arithmetic averages, as requested in the PDF
def summary_stats(panel, rf, benchmark="vw"):
    """Annualised stats. For each portfolio:
    1) take RF monthly series, 2) align by calendar month to portfolio dates,
    3) annualise monthly arithmetic means, 4) Sharpe = annualized mean excess return / annualized volatility."""
    rf_by_month = rf.copy()
    rf_by_month.index = rf_by_month.index.to_period("M")
    rows = []
    bench = panel[benchmark] if benchmark in panel.columns else None
    for col in panel.columns:
        r = panel[col].dropna()
        if not len(r):
            continue
        ann_ret = float(r.mean() * 12.0)
        ann_vol = r.std(ddof=0) * np.sqrt(12.0)
        rf_aligned = rf_by_month.reindex(r.index.to_period("M"))
        valid_rf = rf_aligned.notna().to_numpy()
        ann_rf = float(rf_aligned.dropna().mean() * 12.0) if valid_rf.any() else 0.0
        ret_to_vol = ann_ret / ann_vol if ann_vol > 0 else np.nan
        if ann_vol > 0 and valid_rf.any():
            ann_excess = float((r.to_numpy()[valid_rf] - rf_aligned.to_numpy()[valid_rf]).mean() * 12.0)
            excess_sharpe = ann_excess / ann_vol
        elif ann_vol > 0:
            excess_sharpe = ann_ret / ann_vol
        else:
            excess_sharpe = np.nan
        if bench is not None and col != benchmark:
            diff = (panel[col] - bench).dropna()
            te = float(diff.std(ddof=0) * np.sqrt(12.0)) if len(diff) else np.nan
        else:
            te = np.nan
        gross = (1.0 + r).prod()
        rows.append({
            "portfolio": col,
            "annualized_return": ann_ret,
            "annualized_volatility": ann_vol,
            "annualized_rf": ann_rf,
            "return_to_volatility": ret_to_vol,
            "excess_sharpe_ratio": excess_sharpe,
            "tracking_error_vs_vw": te,
            "minimum_monthly_return": r.min(),
            "maximum_monthly_return": r.max(),
            "cumulative_return": gross - 1.0,
            "terminal_growth": gross,
        })
    return pd.DataFrame(rows)


def plot_cumulative(panel, cols, path, title):
    """Growth of $1 plot - prepend $1 one month before the first return."""
    cum = (1.0 + panel[cols]).cumprod()
    if len(cum):
        first = cum.index.min() - pd.offsets.MonthEnd(1)
        cum = pd.concat([pd.DataFrame(1.0, index=[first], columns=cols), cum])
    fig, ax = plt.subplots(figsize=(10, 6))
    cum.plot(ax=ax, linewidth=2)
    ax.set_title(title); ax.set_xlabel("Date"); ax.set_ylabel("Growth of $1")
    ax.grid(True, linestyle="--", alpha=0.35)
    fig.tight_layout(); fig.savefig(path, dpi=160)
    plt.show()


def plot_annual(df, metric, ports, path, title, ylabel):
    sub = df[df["portfolio"].isin(ports)]
    pivot = sub.pivot(index="year", columns="portfolio", values=metric)[ports]
    fig, ax = plt.subplots(figsize=(10, 6))
    pivot.plot(ax=ax, marker="o", linewidth=2)
    ax.set_title(title); ax.set_xlabel("Allocation year"); ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", alpha=0.35)
    fig.tight_layout(); fig.savefig(path, dpi=160)
    plt.show()

## 4. Part 1 - Minimum-Variance Portfolio (P_mv)

**What we do:** annually rebalanced long-only minimum-variance portfolio. Weights are computed at end-of-Y from the past 10 years of returns and held during Y+1 with intra-year drift (no rebalancing).

**Why it matters:** the MV portfolio is the textbook "low-risk" benchmark. Comparing it to the cap-weighted index shows how much risk an investor can save by simply diversifying optimally.

**Files saved here:** `part1_mv_returns.csv`, `part1_mv_weights.csv`, `part1_performance_mv.csv`, `part1_cumulative_mv.png`.

In [ ]:
# Part 1 tables
mv_returns = monthly[["mv"]].rename(columns={"mv": "mv_returns"})
mv_returns.to_csv(TABLES / "part1_mv_returns.csv", index_label="date")

mv_weights = weights[weights["portfolio"] == "mv"].copy()
mv_weights.to_csv(TABLES / "part1_mv_weights.csv", index=False)

part1_perf = summary_stats(monthly[["vw", "mv"]], rf=rf_monthly, benchmark="vw")
part1_perf.to_csv(TABLES / "part1_performance_mv.csv", index=False)

print("Part 1 - performance summary:")
display(part1_perf)
print("\nMV monthly returns (head):")
display(mv_returns.head())
print("\nMV weights (head):")
display(mv_weights.head(10))

In [ ]:
plot_cumulative(monthly, ["mv"], FIGURES / "part1_cumulative_mv.png",
                "Part 1 - Cumulative growth of $1: minimum-variance portfolio")

## 5. Part 2 - Value-Weighted benchmark (P_vw)

**What we do:** strict Section 2.3 cap-weighted index. Each month, weights are the previous-month market caps normalised to 1; the current-month returns are weighted by those previous-month weights.

**Why it matters:** the VW index is the natural passive benchmark. It tells us what a market-cap investor would have earned with no optimisation, and serves as the reference for tracking error in Parts 3 and 4.

**Files saved here:** `part2_returns_vw.csv`, `part2_performance_vw.csv`, `part2_cumulative_vw.png`, `part2_cumulative_mv_vs_vw.png`.

In [ ]:
vw_returns = monthly[["vw"]].copy()
vw_returns.to_csv(TABLES / "part2_returns_vw.csv", index_label="date")

# Part 2 perf table mirrors the existing format: vw and mv side-by-side
part2_perf = summary_stats(monthly[["vw", "mv"]], rf=rf_monthly, benchmark="vw")
part2_perf.to_csv(TABLES / "part2_performance_vw.csv", index=False)

print("Part 2 - performance summary:")
display(part2_perf)
print("\nVW monthly returns (head):")
display(vw_returns.head())

In [ ]:
plot_cumulative(monthly, ["vw"], FIGURES / "part2_cumulative_vw.png",
                "Part 2 - Cumulative growth of $1: VW benchmark")
plot_cumulative(monthly, ["mv", "vw"], FIGURES / "part2_cumulative_mv_vs_vw.png",
                "Part 2 - Cumulative growth of $1: MV vs VW")

## 6. Part 3 - 50% carbon-footprint reduction

**What we do:** add a carbon-footprint cap equal to 50% of the unconstrained portfolio's CF.
- P_mv_50 = minimum variance s.t. CF(w) <= 0.5 * CF(P_mv).
- P_vw_50 = min tracking error vs P_vw s.t. CF(w) <= 0.5 * CF(P_vw).

**Why it matters:** halving the carbon footprint is a meaningful decarbonisation. The interesting question is whether it is cheap (low TE, similar Sharpe) or expensive (significant risk-adjusted return given up).

Section 3.1 also reports the top-10 WACI and CF contributors of the VW benchmark - i.e. the firms responsible for most of the index's emissions footprint.

In [ ]:
# Section 3.1 - carbon metrics MV vs VW + top10 contributors
metrics_3_1 = annual[annual["portfolio"].isin(["vw_drift", "mv"])].copy()
metrics_3_1["portfolio"] = metrics_3_1["portfolio"].replace({"vw_drift": "vw"})
metrics_3_1.to_csv(TABLES / "part3_carbon_metrics_mv_vw.csv", index=False)
waci_top.to_csv(TABLES / "part3_top10_waci_contributors.csv", index=False)
cf_top.to_csv(TABLES / "part3_top10_cf_contributors.csv", index=False)

print("Section 3.1 - annual carbon metrics MV vs VW:")
display(metrics_3_1.head(12))
print("\nTop-10 WACI contributors (head):")
display(waci_top.head())
print("\nTop-10 CF contributors (head):")
display(cf_top.head())

plot_annual(metrics_3_1, "waci_tco2e_per_musd_revenue", ["vw", "mv"],
            FIGURES / "part3_waci_mv_vs_vw.png",
            "Section 3.1 - WACI: P_mv vs P_vw",
            "WACI (tCO2e per M$ revenue)")
plot_annual(metrics_3_1, "carbon_footprint_tco2e_per_musd_invested", ["vw", "mv"],
            FIGURES / "part3_carbon_footprint_mv_vs_vw.png",
            "Section 3.1 - Carbon footprint: P_mv vs P_vw",
            "CF (tCO2e per M$ invested)")

In [ ]:
# Section 3.2 - MV vs MV(-50%)
monthly[["mv", "mv_50"]].to_csv(TABLES / "part3_returns_mv_carbon50.csv", index_label="date")
summary_mv = summary_stats(monthly[["vw", "mv", "mv_50"]], rf=rf_monthly, benchmark="vw")
summary_mv = summary_mv[summary_mv["portfolio"].isin(["mv", "mv_50"])]
summary_mv.to_csv(TABLES / "part3_summary_mv_vs_mv_carbon50.csv", index=False)
weights[weights["portfolio"].isin(["mv", "mv_50"])].to_csv(
    TABLES / "part3_weights_mv_carbon50.csv", index=False)
slack_table[slack_table["portfolio"] == "mv_50"].to_csv(
    TABLES / "part3_constraint_slack_mv_carbon50.csv", index=False)

print("Section 3.2 - MV vs MV(-50%) summary:")
display(summary_mv)

plot_cumulative(monthly, ["mv", "mv_50"],
                FIGURES / "part3_cumulative_mv_vs_mv_carbon50.png",
                "Section 3.2 - Cumulative growth: MV vs MV(carbon -50%)")
plot_annual(annual, "carbon_footprint_tco2e_per_musd_invested", ["mv", "mv_50"],
            FIGURES / "part3_cf_mv_vs_mv_carbon50.png",
            "Section 3.2 - CF: MV vs MV(carbon -50%)",
            "CF (tCO2e per M$ invested)")
plot_annual(annual, "waci_tco2e_per_musd_revenue", ["mv", "mv_50"],
            FIGURES / "part3_waci_mv_vs_mv_carbon50.png",
            "Section 3.2 - WACI: MV vs MV(carbon -50%)",
            "WACI (tCO2e per M$ revenue)")

In [ ]:
# Section 3.3 - VW vs VW(-50%)
monthly[["vw", "vw_50"]].to_csv(TABLES / "part3_returns_vw_carbon50.csv", index_label="date")
summary_vw = summary_stats(monthly[["vw", "vw_50"]], rf=rf_monthly, benchmark="vw")
summary_vw.to_csv(TABLES / "part3_summary_vw_vs_vw_carbon50.csv", index=False)

weights_vw_view = weights.copy()
weights_vw_view.loc[weights_vw_view["portfolio"] == "vw_drift", "portfolio"] = "vw"
weights_vw_view[weights_vw_view["portfolio"].isin(["vw", "vw_50"])].to_csv(
    TABLES / "part3_weights_vw_carbon50.csv", index=False)
te_table[te_table["portfolio"] == "vw_50"].to_csv(
    TABLES / "part3_tracking_error_vw_carbon50.csv", index=False)
slack_table[slack_table["portfolio"] == "vw_50"].to_csv(
    TABLES / "part3_constraint_slack_vw_carbon50.csv", index=False)

print("Section 3.3 - VW vs VW(-50%) summary:")
display(summary_vw)

plot_cumulative(monthly, ["vw", "vw_50"],
                FIGURES / "part3_cumulative_vw_vs_vw_carbon50.png",
                "Section 3.3 - Cumulative growth: VW vs VW(carbon -50%)")

annual_vw_view = annual.copy()
annual_vw_view["portfolio"] = annual_vw_view["portfolio"].replace({"vw_drift": "vw"})
plot_annual(annual_vw_view, "carbon_footprint_tco2e_per_musd_invested", ["vw", "vw_50"],
            FIGURES / "part3_cf_vw_vs_vw_carbon50.png",
            "Section 3.3 - CF: VW vs VW(carbon -50%)",
            "CF (tCO2e per M$ invested)")
plot_annual(annual_vw_view, "waci_tco2e_per_musd_revenue", ["vw", "vw_50"],
            FIGURES / "part3_waci_vw_vs_vw_carbon50.png",
            "Section 3.3 - WACI: VW vs VW(carbon -50%)",
            "WACI (tCO2e per M$ revenue)")

# ex-ante / ex-post TE plot
te_vw50 = te_table[te_table["portfolio"] == "vw_50"]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(te_vw50["year"], te_vw50["ex_ante_tracking_error_annual"], marker="o", linewidth=2, label="ex-ante")
ax.plot(te_vw50["year"], te_vw50["ex_post_tracking_error_annual"], marker="s", linewidth=2, linestyle="--", label="ex-post")
ax.set_title("Section 3.3 - Annualised tracking error of VW(-50%) vs VW")
ax.set_xlabel("Allocation year"); ax.set_ylabel("Tracking error (annualised)")
ax.grid(True, linestyle="--", alpha=0.35); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "part3_tracking_error_vw_carbon50.png", dpi=160)
plt.show()

## 7. Part 4 - Net-Zero trajectory (P_vw_NZ)

**What we do:** the carbon cap declines by 10% per year from a 2013 anchor equal to CF(P_vw)_{2013}:

$$C_Y = (1 - \theta)^{Y - 2013 + 1} \cdot \text{CF}(P_{vw})_{2013}, \quad \theta = 10\%.$$

For each Y we minimise tracking error vs P_vw subject to $c \cdot w \le C_Y$.

**Why it matters:** this mimics a Paris-aligned investor who commits to a transition trajectory, not a one-off cut. The further out we go, the tighter the cap and the larger the TE: that is the expected cost of an increasingly ambitious decarbonisation pathway.

In [ ]:
monthly[["vw", "vw_nz"]].to_csv(TABLES / "part4_returns_vw_netzero.csv", index_label="date")
joint_monthly = monthly[["vw", "vw_50", "vw_nz"]].copy()
summary_part4 = summary_stats(joint_monthly, rf=rf_monthly, benchmark="vw")
summary_part4.to_csv(TABLES / "part4_summary_vw_vs_carbon50_vs_netzero.csv", index=False)
weights[weights["portfolio"] == "vw_nz"].to_csv(
    TABLES / "part4_weights_vw_netzero.csv", index=False)
path_table.to_csv(TABLES / "part4_netzero_target_vs_realized_cf.csv", index=False)
te_table[te_table["portfolio"] == "vw_nz"].to_csv(
    TABLES / "part4_tracking_error_vw_netzero.csv", index=False)
slack_table[slack_table["portfolio"] == "vw_nz"].to_csv(
    TABLES / "part4_constraint_slack_vw_netzero.csv", index=False)

print("Part 4 - VW vs VW(-50%) vs VW(NZ) summary:")
display(summary_part4)
print("\nPart 4 - net-zero cap path vs realised CF:")
display(path_table)

In [ ]:
# Part 4 figures
plot_cumulative(joint_monthly, ["vw", "vw_50", "vw_nz"],
                FIGURES / "part4_cumulative_vw_vs_carbon50_vs_netzero.png",
                "Section 4 - Cumulative growth: VW vs VW(-50%) vs VW(NZ)")

# CF target vs realised
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(path_table["year"], path_table["carbon_limit_nz"], marker="o", linewidth=2, label="NZ cap target")
ax.plot(path_table["year"], path_table["carbon_footprint_vw_nz"], marker="s", linewidth=2, label="Realised CF (VW(NZ))")
ax.plot(path_table["year"], path_table["carbon_footprint_vw"], marker="^", linewidth=2, linestyle="--", label="CF(VW)")
ax.set_title("Section 4.1 - Net-Zero cap path vs realised carbon footprint")
ax.set_xlabel("Allocation year"); ax.set_ylabel("CF (tCO2e per M$ invested)")
ax.grid(True, linestyle="--", alpha=0.35); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "part4_cf_target_vs_realized.png", dpi=160)
plt.show()

# joint WACI: VW / VW(-50%) / VW(NZ)
annual_view = annual.copy()
annual_view["portfolio"] = annual_view["portfolio"].replace({"vw_drift": "vw"})
waci_panel = annual_view[annual_view["portfolio"].isin(["vw", "vw_50", "vw_nz"])]
plot_annual(waci_panel, "waci_tco2e_per_musd_revenue", ["vw", "vw_50", "vw_nz"],
            FIGURES / "part4_waci_vw_vs_carbon50_vs_netzero.png",
            "Section 4 - WACI: VW vs VW(-50%) vs VW(NZ)",
            "WACI (tCO2e per M$ revenue)")

# TE of VW(NZ)
te_nz = te_table[te_table["portfolio"] == "vw_nz"]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(te_nz["year"], te_nz["ex_ante_tracking_error_annual"], marker="o", linewidth=2, label="ex-ante")
ax.plot(te_nz["year"], te_nz["ex_post_tracking_error_annual"], marker="s", linewidth=2, linestyle="--", label="ex-post")
ax.set_title("Section 4 - Annualised tracking error of VW(NZ) vs VW")
ax.set_xlabel("Allocation year"); ax.set_ylabel("Tracking error (annualised)")
ax.grid(True, linestyle="--", alpha=0.35); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES / "part4_tracking_error_vw_netzero.png", dpi=160)
plt.show()

## 8. Final performance summary

### Sharpe ratio (aligned RF)

Every Sharpe ratio in this notebook is built from annualized arithmetic monthly averages, with the risk-free rate aligned to the same calendar month as the portfolio returns:

1. take the monthly RF series,
2. align it to portfolio returns by calendar month, so trading month-ends and calendar month-ends still match,
3. annualise the monthly portfolio mean, RF mean, and excess-return mean,
4. Sharpe = annualized mean excess return / annualized volatility.

The validation cell below shows the date range, number of observations, RF matches, and aligned RF for every portfolio. This makes the alignment fully verifiable.

In [ ]:
# Validation cell: shows that RF is aligned per portfolio by calendar month
rf_by_month = rf_monthly.copy()
rf_by_month.index = rf_by_month.index.to_period("M")
validation_rows = []
for col in monthly.columns:
    r = monthly[col].dropna()
    if not len(r):
        continue
    rf_aligned = rf_by_month.reindex(r.index.to_period("M"))
    valid_rf = rf_aligned.notna().to_numpy()
    ann_ret = float(r.mean() * 12.0)
    ann_vol = r.std(ddof=0) * np.sqrt(12.0)
    ann_rf = float(rf_aligned.dropna().mean() * 12.0) if valid_rf.any() else 0.0
    if ann_vol > 0 and valid_rf.any():
        ann_excess = float((r.to_numpy()[valid_rf] - rf_aligned.to_numpy()[valid_rf]).mean() * 12.0)
        sharpe = ann_excess / ann_vol
    elif ann_vol > 0:
        sharpe = ann_ret / ann_vol
    else:
        sharpe = np.nan
    validation_rows.append({
        "portfolio": col,
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "n_obs": int(len(r)),
        "n_rf_matches": int(valid_rf.sum()),
        "annualized_return": ann_ret,
        "annualized_volatility": ann_vol,
        "aligned_annualized_rf": ann_rf,
        "sharpe_ratio": sharpe,
    })
validation_df = pd.DataFrame(validation_rows)
print("Validation - RF aligned per portfolio (portfolio evaluation window only):")
display(validation_df)

In [ ]:
# Final headline summary across all five portfolios
final_summary = summary_stats(monthly[["vw", "mv", "mv_50", "vw_50", "vw_nz"]],
                              rf=rf_monthly, benchmark="vw")
final_summary.to_csv(TABLES / "final_performance_summary.csv", index=False)
print("Final performance summary (saved to outputs/tables/final_performance_summary.csv):")
display(final_summary)

portfolio_file_index = pd.DataFrame([
    {"portfolio": "vw", "description": "strict monthly-rebalanced value-weighted benchmark", "main_return_file": "part2_returns_vw.csv"},
    {"portfolio": "vw_drift", "description": "annual value-weighted allocation with intra-year drift", "main_return_file": "all_portfolio_monthly_returns.csv"},
    {"portfolio": "mv", "description": "long-only minimum-variance portfolio", "main_return_file": "part1_mv_returns.csv"},
    {"portfolio": "mv_50", "description": "minimum variance with 50% carbon-footprint cap", "main_return_file": "part3_returns_mv_carbon50.csv"},
    {"portfolio": "vw_50", "description": "tracking-error portfolio with 50% carbon-footprint cap", "main_return_file": "part3_returns_vw_carbon50.csv"},
    {"portfolio": "vw_nz", "description": "tracking-error portfolio on the Net-Zero carbon path", "main_return_file": "part4_returns_vw_netzero.csv"},
])
portfolio_file_index.to_csv(TABLES / "portfolio_file_index.csv", index=False)

output_files = file_inventory(OUTPUTS)
output_files.to_csv(TABLES / "output_file_inventory.csv", index=False)
print("Portfolio construction index:")
display(portfolio_file_index)
print("Output files generated by this notebook:")
display(output_files)

## 9. Short conclusion

- **MV vs VW.** The minimum-variance portfolio realises lower volatility *and* a higher Sharpe ratio than the cap-weighted index, confirming the textbook MV diversification benefit on the Pacific universe.
- **50% carbon cut.** Both MV(-50%) and VW(-50%) are feasible every year. They decarbonise the portfolio substantially without hurting risk-adjusted performance: the VW(-50%) tracking error stays small, which is good news for benchmark-relative investors.
- **Net-Zero trajectory.** The 10%-per-year cap binds harder over time. Realised CF closely follows the cap, the WACI drops more sharply than under the static 50% cut, and the tracking error rises as the constraint tightens - the expected economic cost of a credible decarbonisation pathway.
- **Bottom line.** A passive cap-weighted investor can decarbonise materially at low TE; an aggressive Net-Zero trajectory eventually requires accepting some active risk.

### Use of Large Language Models (LLMs)

OpenAI Codex was used as a coding assistant to inspect the notebook, compare the implementation with the project PDF, debug date-alignment and summary-statistic issues, and improve code readability. The group remains responsible for understanding the methodology, validating the results, interpreting the findings, and ensuring academic integrity.